# Лабораторная работа №4 — FastSpeech2 на RUSLAN

## пути и файлы

In [ ]:
import os, sys, re, shlex, shutil, subprocess
from pathlib import Path

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi не нашёл GPU")

ROOT = Path.home() / "Документы/labs/tts_labs_itmo"
LAB = ROOT / "labs/lab4_acoustic"
DATA = ROOT / "data"
COMMON = ROOT / "common"
WAV_LOCAL = DATA / "ruslan_dataset/RUSLAN"

GPU 0: NVIDIA GeForce GTX 1080 Ti (UUID: GPU-a49f351b-f160-3c5c-f7ae-9f7495eb710c)

OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/labs/lab4_acoustic/train_fastspeech2.py
OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/labs/lab4_acoustic/evaluate.py
OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/labs/lab4_acoustic/prepare_mfa_durations.py
OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/labs/lab4_acoustic/data/eval_sentences.csv
OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/common/phones_mfa.txt
OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/data/metadata_phones.csv
OK   /home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/data/metadata_phones_gt.csv


## разбивка данных train/test
- правило «id делится на 5»

In [2]:
def read_rows(p):
    return [l for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]

rows_g2p = read_rows(DATA / "metadata_phones.csv")
rows_gt = read_rows(DATA / "metadata_phones_gt.csv")
eval_rows = read_rows(LAB / "data/eval_sentences.csv")

for name, rows in (("metadata_phones.csv (g2p)", rows_g2p), ("metadata_phones_gt.csv (gt)", rows_gt), ("eval_sentences.csv", eval_rows)):
    print(f"{name}: {len(rows)} строк")
    print("   ", rows[0][:160])

ids_gt = [r.split("|")[0] for r in rows_gt]
have = sum((WAV_LOCAL / f"{i}.wav").is_file() for i in ids_gt)
print(f"\nid из gt-метаданных, для которых есть wav: {have}/{len(ids_gt)}")

metadata_phones.csv (g2p): 20578 строк
    000000_RUSLAN|С тревожным чувством берусь я за перо.|s̪ t̪ rʲ ɪ v o ʐ n̪ ɨ m tɕ u s̪ t̪ v ə m bʲ ɪ r u sʲ j a z̪ ə pʲ ɪ r o <SIL>
metadata_phones_gt.csv (gt): 20578 строк
    000000_RUSLAN|С тревожным чувством берусь я за перо.|s̪ t̪ rʲ ɪ v o ʐ n̪ ɨ m tɕ u s̪ t̪ v ə m bʲ ɪ r u sʲ j a z̪ ə pʲ ɪ r o <SIL>
eval_sentences.csv: 120 строк
    000020_RUSLAN|Ведь представления, которые оно рождает, безграничны до нуля.|vʲ e tʲ p rʲ ɪ t̪ s̪ t̪ ɐ v ʎ e ɲ ɪ j ə <SIL> k ə t̪ ɐ r ɨ j e ɐ n̪ o r ɐ ʐ d̪ a j ɪ

id из gt-метаданных, для которых есть wav: 20578/20578


In [3]:
rid = lambda r: r.split("|")[0]
has_wav = lambda r: (WAV_LOCAL / f"{rid(r)}.wav").is_file()

# 1. пропускаем строки без аудио
gt_ok = [r for r in rows_gt if has_wav(r)]
g2p_ok = [r for r in rows_g2p if has_wav(r)]
print(f"gt : {len(gt_ok)}/{len(rows_gt)} со звуком (пропущено {len(rows_gt) - len(gt_ok)})")
print(f"g2p: {len(g2p_ok)}/{len(rows_g2p)} со звуком (пропущено {len(rows_g2p) - len(g2p_ok)})")

# 2. что с eval-фразами
eval_set = {rid(r) for r in eval_rows}
meta_ids = {rid(r) for r in rows_gt}
wav_ids = {rid(r) for r in gt_ok}
target = eval_set & wav_ids
print(f"\neval-фраз: {len(eval_set)} | есть в gt-метаданных: {len(eval_set & meta_ids)} | и со звуком: {len(target)}")
print("примеры id  eval:", sorted(eval_set)[:3], "| метаданные:", [rid(r) for r in rows_gt[:3]])
assert target, "ни одной eval-фразы нет в метаданных со звуком: формат id в eval_sentences.csv и в metadata не совпадает"

cand = {rid(r) for k, r in enumerate(rows_gt) if k % 5 == 0}

# eval-фразы всегда в test
test_ids = cand | target
print(f"Добавлено в test сверх правила: {len(target - cand)}")

def split(rows, tag):
    train = [r for r in rows if rid(r) not in test_ids]
    test = [r for r in rows if rid(r) in test_ids]
    (DATA / f"metadata_{tag}_train.csv").write_text("\n".join(train) + "\n", encoding="utf-8")
    (DATA / f"metadata_{tag}_test.csv").write_text("\n".join(test) + "\n", encoding="utf-8")
    print(f"{tag}: train {len(train)}, test {len(test)}")

split(gt_ok, "phones_gt")
split(g2p_ok, "phones")

gt : 20578/20578 со звуком (пропущено 0)
g2p: 20578/20578 со звуком (пропущено 0)

eval-фраз: 120 | есть в gt-метаданных: 118 | и со звуком: 118
примеры id  eval: ['000020_RUSLAN', '000080_RUSLAN', '000395_RUSLAN'] | метаданные: ['000000_RUSLAN', '000001_RUSLAN', '000002_RUSLAN']
Добавлено в test сверх правила: 90
phones_gt: train 16372, test 4206
phones: train 16372, test 4206


In [4]:
def phones_by_id(rows):
    return {r.split("|")[0]: r.split("|")[2] for r in rows if r.count("|") >= 2}

d_gt, d_g2p = phones_by_id(rows_gt), phones_by_id(rows_g2p)
ev = phones_by_id(eval_rows)
print("фонемы eval_sentences совпадают с gt:", sum(ev[i] == d_gt.get(i) for i in ev), "| с g2p:", sum(ev[i] == d_g2p.get(i) for i in ev), "| всего", len(ev))

for tag, d in (("gt", d_gt), ("g2p", d_g2p)):
    out = []
    for r in eval_rows:
        c = r.split("|")
        out.append("|".join([c[0], c[1], d.get(c[0], c[2])]))
    (LAB / f"data/eval_sentences_{tag}.csv").write_text("\n".join(out) + "\n", encoding="utf-8")
print("записаны eval_sentences_gt.csv и eval_sentences_g2p.csv")

фонемы eval_sentences совпадают с gt: 88 | с g2p: 60 | всего 120
записаны eval_sentences_gt.csv и eval_sentences_g2p.csv


## обучение

- лог в `runs/`, ждём в логе `pre-flight ok`.
- если не хватит памяти GPU, добавить `extra=["--batch-size", "16"]` и `lr="3e-4"`.

In [16]:
def launch(run_name, metadata, lr="1e-3", extra=()):
    log = LAB / "runs" / f"{run_name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "train_fastspeech2.py",
           "--run-name", run_name, "--lr", lr,
           "--metadata", str(metadata),
           "--wavs", str(WAV_LOCAL),
           "--phones", str(COMMON / "phones_mfa.txt"),
           "--output", str(LAB / "runs"), *extra]
    print(shlex.join(cmd))
    return subprocess.Popen(cmd, cwd=LAB, stdout=open(log, "a"),
                            stderr=subprocess.STDOUT, start_new_session=True)

proc = launch("baseline_gt", DATA / "metadata_phones_gt_train.csv", lr="5e-3")

'/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/.venv/bin/python' train_fastspeech2.py --run-name baseline_gt --lr 5e-3 --metadata '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/data/metadata_phones_gt_train.csv' --wavs '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/data/ruslan_dataset/RUSLAN' --phones '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/common/phones_mfa.txt' --output '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/labs/lab4_acoustic/runs'


In [27]:
r = subprocess.run(["pgrep", "-af", "train_fastspeech2"], capture_output=True, text=True)
print(r.stdout or "процесс обучения не найден")
!tail -n 30 "{LAB}/runs/baseline_gt.log"

процесс обучения не найден
     | > grad_norm: tensor(2.0824, device='cuda:0')  (tensor(1.8652, device='cuda:0'))
     | > step_time: 0.4963  (0.43904043619449323)
     | > loader_time: 0.0113  (0.15152485095537624)


   --> TIME: 2026-11-03 11:13:17 -- STEP: 154/495 -- GLOBAL_STEP: 50250
     | > current_lr: 0.0001275  (0.00012749999999999974)
     | > loss_spec: 0.4435805678367615  (0.42726120372097215)
     | > loss_dur: 0.007974728010594845  (0.006663014329123226)
     | > loss_aligner: 1.344956398010254  (1.0736267322843718)
     | > loss_binary_alignment: 0.08593206107616425  (0.0638507433574308)
     | > loss: 1.924133539199829  (1.6023787794175084)
     | > duration_error: 1.3939781188964844  (1.4107491590759966)
     | > grad_norm: tensor(1.5514, device='cuda:0')  (tensor(1.7814, device='cuda:0'))
     | > step_time: 0.5497  (0.4667386760959378)
     | > loader_time: 0.0075  (0.17218399357486086)


   --> TIME: 2026-11-03 11:13:56 -- STEP: 204/495 -- GLOBAL_STEP: 50300
     | 

In [ ]:
tb = Path(sys.executable).parent / "tensorboard"
print(tb, "| есть" if tb.exists() else "| НЕТ")
os.environ["TENSORBOARD_BINARY"] = str(tb)

%reload_ext tensorboard
%tensorboard --logdir "{LAB}/runs"

/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/.venv/bin/tensorboard | есть


Launching TensorBoard...

### Второй обязательный запуск — g2p-фонемы
Те же параметры, меняется только файл метаданных. Запускайте после того, как первый закончился (или остановите его).

In [28]:
proc = launch("baseline_g2p", DATA / "metadata_phones_train.csv", lr="5e-3")

'/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/.venv/bin/python' train_fastspeech2.py --run-name baseline_g2p --lr 5e-3 --metadata '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/data/metadata_phones_train.csv' --wavs '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/data/ruslan_dataset/RUSLAN' --phones '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/common/phones_mfa.txt' --output '/home/ext-yankin@ad.speechpro.com/Документы/labs/tts_labs_itmo/labs/lab4_acoustic/runs'


In [31]:
r = subprocess.run(["pgrep", "-af", "train_fastspeech2"], capture_output=True, text=True)
print(r.stdout or "процесс обучения не найден")
!tail -n 30 "{LAB}/runs/baseline_gt.log"

процесс обучения не найден
     | > grad_norm: tensor(2.0824, device='cuda:0')  (tensor(1.8652, device='cuda:0'))
     | > step_time: 0.4963  (0.43904043619449323)
     | > loader_time: 0.0113  (0.15152485095537624)


   --> TIME: 2026-11-03 11:13:17 -- STEP: 154/495 -- GLOBAL_STEP: 50250
     | > current_lr: 0.0001275  (0.00012749999999999974)
     | > loss_spec: 0.4435805678367615  (0.42726120372097215)
     | > loss_dur: 0.007974728010594845  (0.006663014329123226)
     | > loss_aligner: 1.344956398010254  (1.0736267322843718)
     | > loss_binary_alignment: 0.08593206107616425  (0.0638507433574308)
     | > loss: 1.924133539199829  (1.6023787794175084)
     | > duration_error: 1.3939781188964844  (1.4107491590759966)
     | > grad_norm: tensor(1.5514, device='cuda:0')  (tensor(1.7814, device='cuda:0'))
     | > step_time: 0.5497  (0.4667386760959378)
     | > loader_time: 0.0075  (0.17218399357486086)


   --> TIME: 2026-11-03 11:13:56 -- STEP: 204/495 -- GLOBAL_STEP: 50300
     | 

## оценка

In [ ]:
def evaluate(run_name, eval_tag, train_meta, extra=()):
    cmd = [sys.executable, "evaluate.py", "--run", f"runs/{run_name}",
           "--eval-set", str(LAB / f"data/eval_sentences_{eval_tag}.csv"),
           "--wavs", str(WAV_LOCAL), "--metadata", str(train_meta), *extra]
    r = subprocess.run(cmd, cwd=LAB, capture_output=True, text=True)
    print(r.stdout[-5000:])
    if r.returncode:
        print("STDERR:", r.stderr[-4000:])

# быстрая проверка extra=["--limit", "5"]
evaluate("baseline_gt", "gt", DATA / "metadata_phones_gt_train.csv")

69   MCD  24.06 dB
  006100_RUSLAN  frames  446 ->  537   MCD  31.26 dB
  006170_RUSLAN  frames  391 ->  404   MCD  28.47 dB
  006175_RUSLAN  frames  456 ->  482   MCD  29.61 dB
  006250_RUSLAN  frames  462 ->  517   MCD  26.71 dB
  006335_RUSLAN  frames  451 ->  482   MCD  32.73 dB
  006515_RUSLAN  frames  573 ->  620   MCD  28.82 dB
  006650_RUSLAN  frames  452 ->  516   MCD  28.37 dB
  006725_RUSLAN  frames  206 ->  226   MCD  29.10 dB
  006890_RUSLAN  frames  462 ->  515   MCD  32.09 dB
  006930_RUSLAN  frames  358 ->  424   MCD  30.28 dB
  007095_RUSLAN  frames  342 ->  336   MCD  30.02 dB
  007155_RUSLAN  frames  597 ->  572   MCD  28.37 dB
  007485_RUSLAN  frames  400 ->  436   MCD  32.75 dB
  007690_RUSLAN  frames  428 ->  490   MCD  29.35 dB
  008010_RUSLAN  frames  415 ->  442   MCD  28.86 dB
  008375_RUSLAN  frames  540 ->  523   MCD  28.04 dB
  008585_RUSLAN  frames  494 ->  542   MCD  27.29 dB
  008885_RUSLAN  frames  487 ->  544   MCD  26.88 dB
  008915_RUSLAN  frames  52

In [33]:
evaluate("baseline_g2p", "g2p", DATA / "metadata_phones_train.csv")

   MCD  23.66 dB
  006100_RUSLAN  frames  446 ->  507   MCD  28.27 dB
  006170_RUSLAN  frames  391 ->  405   MCD  28.82 dB
  006175_RUSLAN  frames  456 ->  465   MCD  26.97 dB
  006250_RUSLAN  frames  462 ->  512   MCD  26.83 dB
  006335_RUSLAN  frames  451 ->  480   MCD  31.23 dB
  006515_RUSLAN  frames  573 ->  621   MCD  30.90 dB
  006650_RUSLAN  frames  452 ->  507   MCD  30.70 dB
  006725_RUSLAN  frames  206 ->  234   MCD  28.65 dB
  006890_RUSLAN  frames  462 ->  489   MCD  32.29 dB
  006930_RUSLAN  frames  358 ->  410   MCD  31.34 dB
  007095_RUSLAN  frames  342 ->  339   MCD  31.29 dB
  007155_RUSLAN  frames  597 ->  574   MCD  28.92 dB
  007485_RUSLAN  frames  400 ->  438   MCD  30.07 dB
  007690_RUSLAN  frames  428 ->  483   MCD  31.09 dB
  008010_RUSLAN  frames  415 ->  459   MCD  27.81 dB
  008375_RUSLAN  frames  540 ->  564   MCD  29.91 dB
  008585_RUSLAN  frames  494 ->  535   MCD  28.75 dB
  008885_RUSLAN  frames  487 ->  538   MCD  27.63 dB
  008915_RUSLAN  frames  522 

In [38]:
import random
import IPython.display as ipd

run_eval = next((LAB / "runs" / "baseline_gt").rglob("eval"))
wav_files = sorted((run_eval / "natural").glob("*.wav"))

if wav_files:
    random_file = random.choice(wav_files)
    uid = random_file.stem
    print(f"Playing audio for UID: {uid}\n")
    for kind in ("natural", "copysynth", "model"):
        print(f"--- {kind} ---")
        audio_path = run_eval / kind / f"{uid}.wav"
        if audio_path.exists():
            ipd.display(ipd.Audio(str(audio_path)))
        else:
            print(f"Файл не найден: {audio_path.name}")

Playing audio for UID: 009655_RUSLAN

--- natural ---


--- copysynth ---


--- model ---


## 9. Что сдавать
Из `labs/lab4_acoustic/runs/<лучшая_модель>/`: `config.json`, логи, лучший чекпоинт, `eval/` (wav трёх видов и `mcd.csv`); плюс отчёт `report.*` в `labs/lab4_acoustic/`.